In [19]:
import pandas as pd
import numpy as np
import os 
from langchain.document_loaders import PyPDFLoader, UnstructuredAPIFileIOLoader, PyPDFium2Loader
from langchain.document_loaders import PyPDFDirectoryLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pathlib import Path
import random

In [20]:
data_dir = 'TEXT_data'
inputdirectory = Path(f"./{data_dir}")
out_dir = data_dir
outputdirectory = Path(f"./TEXT_data/output")

# Loading Data

In [21]:
loader = DirectoryLoader(inputdirectory, show_progress=True)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150,
    length_function = len,
    is_separator_regex = False,
)

pages = splitter.split_documents(documents)
print("Number of chunks =", len(pages))
print(pages[3].page_content)

100%|██████████| 1/1 [00:00<00:00,  1.95it/s]

Number of chunks = 81
The execution of the Project Risk Management process is dealt with in subsequent chapters of this practice standard and so is not discussed here. 2.2 Definition of Project Risk The word “risk” is used in many ways in everyday language and in various specialist disciplines. Its use in the PMBOK Guide – Fourth Edition is as follows: Project risk is an uncertain event or condition that, if it occurs, has a positive or a negative effect on a project’s objectives. This Definition includes two key dimensions of risk: uncertainty and effect on a project’s objectives. When assessing the importance of a project risk, these two dimensions must both be considered. The uncertainty dimension may be described using the term “probability” and the effect may be called “impact” (though other descriptors are possible, such as “likelihood” and “consequence”). The Definition of risk includes both distinct events which are uncertain but can be clearly described, and more general condi

#### Create a dataframe of all the chunks

In [23]:
from helpers.df_helpers import documents2Dataframe
df = documents2Dataframe(pages)
print(df.shape)
df.head()

(81, 3)


,text,source,chunk_id
0,CHAPTER 1 INTRODUCTION Project Management Inst...,TEXT_data\risk_management_final.txt,36c6789384b5483a8991ed1052c5e4b5
1,While tools and techniques are constantly evol...,TEXT_data\risk_management_final.txt,782faaacdd9b406e951d56b4a27a90b7
2,CHAPTER 1 − INTRODUCTION 1.2 Project Risk Mana...,TEXT_data\risk_management_final.txt,3ed670feb91f4be5bc4b956bfb938497
3,The execution of the Project Risk Management p...,TEXT_data\risk_management_final.txt,6465172d9a58460793682c555aa157e2
4,This allows for the gain of synergies and efﬁ ...,TEXT_data\risk_management_final.txt,bf42d83208804dba90da47c449354dc8


In [24]:
from helpers.df_helpers import df2Graph
from helpers.df_helpers import graph2Df

In [25]:
concepts_list = df2Graph(df, model='mistral')

 [
      {
          "node_1": "Project Risk Management processes",
          "node_2": "PMBOK Guide – Fourth Edition",
          "edge": "The Project Risk Management processes are defined and described in this practice standard, which is based on the PMBOK Guide – Fourth Edition."
      },
      {
          "node_1": "Plan Risk Management",
          "node_2": "Chapter 1 (Organizational Context)",
          "edge": "Chapter 1 provides an introduction to Plan Risk Management, one of the six processes in Project Risk Management."
      },
      {
          "node_1": "Perform Qualitative Risk Analysis",
          "node_2": "Chapter 2 (Performance Domain)",
          "edge": "Chapter 2 provides a detailed description of Perform Qualitative Risk Analysis, one of the six processes in Project Risk Management."
      },
      {
          "node_1": "Plan Risk Responses",
          "node_2": "Chapter 3 (Planning Domain)",
          "edge": "Chapter 3 provides a detailed description of Plan Risk

In [51]:
dfg1 = graph2Df(concepts_list)
if not os.path.exists(outputdirectory):
        os.makedirs(outputdirectory)

In [52]:
dfg1.to_csv(outputdirectory/"graph.csv", sep="|", index=False)
df.to_csv(outputdirectory/"chunks.csv", sep="|", index=False)

In [34]:
dfg1 = pd.read_csv('TEXT_data/output/graph.csv', sep='|')
df = pd.read_csv('TEXT_data/output/chunks.csv', sep='|')

In [53]:
dfg1.replace("", np.nan, inplace=True)
dfg1.dropna(subset=["node_1", "node_2", 'edge'], inplace=True)
dfg1['count'] = 1

In [54]:
dfg1.shape

(783, 5)

In [55]:
dfg1.head(5)

,node_1,node_2,edge,chunk_id,count
0,project risk management processes,pmbok guide – fourth edition,The Project Risk Management processes are defi...,36c6789384b5483a8991ed1052c5e4b5,1
1,plan risk management,chapter 1 (organizational context),Chapter 1 provides an introduction to Plan Ris...,36c6789384b5483a8991ed1052c5e4b5,1
2,perform qualitative risk analysis,chapter 2 (performance domain),Chapter 2 provides a detailed description of P...,36c6789384b5483a8991ed1052c5e4b5,1
3,plan risk responses,chapter 3 (planning domain),Chapter 3 provides a detailed description of P...,36c6789384b5483a8991ed1052c5e4b5,1
4,purpose and objectives of each process,each chapter,Each chapter in this practice standard address...,36c6789384b5483a8991ed1052c5e4b5,1


In [56]:
def contextual_proximity(df: pd.DataFrame) -> pd.DataFrame:
    dfg_long = pd.melt(
        df, id_vars=["chunk_id"], value_vars=["node_1","node_2"], value_name="node"
    )

    dfg_long.drop(columns = ["variable"], inplace=True)
    dfg_wide = pd.merge(dfg_long, dfg_long, on="chunk_id", suffixes=("_1", "_2"))
    self_loops_drop  = dfg_wide[dfg_wide["node_1"]==dfg_wide["node_2"]].index
    dfg2 = dfg_wide.drop(index = self_loops_drop).reset_index(drop=True)

    dfg2=(
        dfg2.groupby(["node_1", "node_2"])
        .agg({"chunk_id":[",".join,"count"]})
        .reset_index()
    )

    dfg2.columns = ["node_1", "node_2", "chunk_id", "count"]
    dfg2.replace("", np.nan, inplace=True)
    dfg2.dropna(subset=["node_1", "node_2"], inplace=True)

    dfg2 = dfg2[dfg2["count"] != 1]
    dfg2["edge"] = "contextual_proximity"
    return dfg2

dfg2 = contextual_proximity(dfg1)
dfg2.tail()

,node_1,node_2,chunk_id,count,edge
19835,work breakdown structure (wbs),system dynamics analysis,"59786994019441008498f8cd0aec6917,5978699401944...",3,contextual_proximity
19836,work breakdown structure (wbs),system dynamics system (sd),"59786994019441008498f8cd0aec6917,5978699401944...",5,contextual_proximity
19848,"workshops, interviews, or questionnaires",data gathering tools,"51a83bad1dc544f68546d703a7e816e6,51a83bad1dc54...",2,contextual_proximity
19851,"workshops, interviews, or questionnaires","project planning, budgeting, and scheduling","51a83bad1dc544f68546d703a7e816e6,51a83bad1dc54...",5,contextual_proximity
19853,"workshops, interviews, or questionnaires",quantitative analysis results,"51a83bad1dc544f68546d703a7e816e6,51a83bad1dc54...",5,contextual_proximity


In [58]:
dfg = pd.concat([dfg1, dfg2], axis = 0)
dfg = (
    dfg.groupby(["node_1", "node_2"])
    .agg({"chunk_id": ",".join, "edge": ','.join, 'count':'sum'})
    .reset_index()
)

In [60]:
dfg.head(5)

,node_1,node_2,chunk_id,edge,count
0,'confidence',data gathering tools,"51a83bad1dc544f68546d703a7e816e6,51a83bad1dc54...",contextual_proximity,2
1,'confidence',"project planning, budgeting, and scheduling","51a83bad1dc544f68546d703a7e816e6,51a83bad1dc54...",contextual_proximity,5
2,'confidence',quantitative analysis results,"51a83bad1dc544f68546d703a7e816e6,51a83bad1dc54...",contextual_proximity,5
3,'correlation between risks',data gathering tools,"51a83bad1dc544f68546d703a7e816e6,51a83bad1dc54...",contextual_proximity,2
4,'correlation between risks',"project planning, budgeting, and scheduling","51a83bad1dc544f68546d703a7e816e6,51a83bad1dc54...",contextual_proximity,5


# Calculate the NetworkX Graph

In [62]:
nodes = pd.concat([dfg['node_1'], dfg['node_2']], axis = 0).unique()
nodes.shape

(993,)

In [70]:
import networkx as nx
G = nx.Graph()

for node in nodes:
    G.add_node(
        str(node)
    )

for index, row in dfg.iterrows():
    G.add_edge(
        str(row["node_1"]),
        str(row["node_2"]),
        title=row["edge"],
        weight=row['count']
    )

#### Calculate communities for coloring nodes

In [71]:
communities_generator = nx.community.girvan_newman(G)
top_level_communities = next(communities_generator)
next_level_communities = next(communities_generator)
communities = sorted(map(sorted, next_level_communities))

print("Number of Communities:", len(communities))
print(communities)

Number of Communities: 39
[["'confidence'", "'correlation between risks'", "'expected value of a project decision'", "'identity or location within the project model of the most important risks'", "'probability distribution of its potential impacts on cost or time'", "'probability of a risk occurring'", "'probability of achieving a project objective such as finishing on time or within budget'", "'set up a business structure in which the customer and the supplier share the risk'", "'technical feasibility; actions; and balance between overall impact of the response on the project objectives and the improvement in'", 'assessment of historical data', 'data gathering tools', 'probability distribution of project completion dates or total costs', 'project communication planning', 'project planning, budgeting, and scheduling', 'pursue the project despite a risk exposure that exceeds the desired level', 'quantitative analysis results', 'workshops, interviews, or questionnaires'], ["'high risk'",

#### Create a dataframe for community colors

In [72]:
import seaborn as sns
palette = "hls"

def colors2Community(communities) -> pd.DataFrame:

    p=sns.color_palette(palette, len(communities)).as_hex()
    random.shuffle(p)
    rows = []
    group=0

    for community in communities:
        color = p.pop()
        group+=1
        
        for node in community:
            rows += [{"node": node, "color": color, "group": group}]
    
    df_colors = pd.DataFrame(rows)
    return df_colors

In [73]:
colors = colors2Community(communities)
colors.head(5)

,node,color,group
0,'confidence',#57db5f,1
1,'correlation between risks',#57db5f,1
2,'expected value of a project decision',#57db5f,1
3,'identity or location within the project model...,#57db5f,1
4,'probability distribution of its potential imp...,#57db5f,1


#### Add colors to the graph

In [74]:
for index, row in colors.iterrows():
    G.nodes[row['node']]['group'] = row['group']
    G.nodes[row['node']]['color'] = row['color']
    G.nodes[row['node']]['size'] = G.degree[row['node']]

In [76]:
from pyvis.network import Network 

In [80]:
graph_output_directory = "./docs/index.html"

net = Network (
    notebook = False,
    cdn_resources = "remote",
    height = "900px",
    width = "100%",
    select_menu = True,
    filter_menu = False,
)

In [81]:
net.from_nx(G)
net.force_atlas_2based(central_gravity = 0.015, gravity = -31)
net.show_buttons(filter_= ["physics"])
net.show(graph_output_directory, notebook=False)

./docs/index.html
